# IMPORTANT

Run notebooks/cleaning/download_and_clean_data.ipynb first if dataset fetch fails.

In [1]:
!pip install Faker

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 8.9 MB/s  0:00:00



[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import faker
import numpy as np
from scipy import stats
import pandas as pd
from pathlib import Path
import os
import random
# after seeing what Faker does I have a sneaking suspicion we do not need it. Still, because the requirements have it as a guide, might as well import it for auld lang syne.

In [11]:
# importing notebook

PROJECT_ROOT = Path(os.getcwd()).parents[1]

raw_dataset_path = os.path.join(PROJECT_ROOT, 'data', 'raw', 'raw_df.parquet.gzip')
df = pd.read_parquet(raw_dataset_path)
print(df.head())
print(len(df))

   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  
6362620


In [17]:
# |user_agent|object|NaN|1 of 6 user agents|
# Different distributions for fraud and non-fraud
# User agents: Android, iOS, Windows, MacOS, Linux, Others

agents_list = ['Android', 'iOS', 'Windows', 'MacOS', 'Linux', 'Others']
non_fraud_weights = [0.35525, 0.13475, 0.37975, 0.09188, 0.01837, 0.02]
# Assuming equal split 49-49-2 for mobile-desktop-others, according to global figures, Android ~ 72.5% to iOS ~ 27.5%
# Desktop: 62% Windows, 15% MacOS, 3% Linux, the rest others => extrapolate out
fraud_weights = [0.325, 0.05, 0.25, 0.015, 0.225, 0.135]
# Slightly different to account for agent sources that can be easily used by attackers such as Linux

df['user_agent'] = df['isFraud'].apply(lambda x: np.random.choice(agents_list, size=1, p=non_fraud_weights)[0] if x == 0 else
                                       np.random.choice(agents_list, size=1, p=fraud_weights)[0])

print(df.head())

   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  \
0  M1979787155             0.0             0.0        0               0   
1  M2044282225             0.0             0.0        0               0   
2   C553264065             0.0             0.0        1               0   
3    C38997010         21182.0             0.0        1               0   
4  M1230701703             0.0             0.0        0               0   

  user_agent  
0    Android  
1    Windows  
2    Windows  
3     Others  
4    Android  


In [19]:
# |failed_attempts|int|NaN|0-6|
# Sample from distribution with mean = 0.065 to simulate 5-8% fail rate normally 
# Up to mean = 0.26 (x4) to simulate potential existing antifraud/irregularities in fraud transactions

# Modelled using Poisson distribution because it looks right (just kidding, look at the markdown for rationale)

df['failed_attempts'] = df['isFraud'].apply(lambda x: stats.poisson.rvs(0.065, size=1)[0] if x == 0 else
                                       stats.poisson.rvs(0.26, size=1)[0])

print(df.head())

   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  \
0  M1979787155             0.0             0.0        0               0   
1  M2044282225             0.0             0.0        0               0   
2   C553264065             0.0             0.0        1               0   
3    C38997010         21182.0             0.0        1               0   
4  M1230701703             0.0             0.0        0               0   

  user_agent  failed_attempts  
0    Android                0  
1    Windows                0  
2 

In [21]:
# |distance_to_billing|float64|km|0-12000
# Most transactions local => 0
# Some non-fraud transactions are to neighbouring countries around 750-2000km, very few if any >2000
# Fraud transactions have much higher occurance of higher values

def distance_to_billing_gen(fraud_status):
    # np.random.default_rng().integers(low=750, high=2000, size=1)[0]
    if fraud_status == 0:
        choice = np.random.choice([0, 1, 2], size=1, p=[0.65, 0.30, 0.05])
    else:
        choice = np.random.choice([0, 1, 2], size=1, p=[0.45, 0.30, 0.25])
    if choice == 0:
        return 0
    elif choice == 1:
        return np.random.default_rng().integers(low=750, high=2000, size=1)[0]
    else:
        return np.random.default_rng().integers(low=2067, high=12000, size=1)[0]

df['distance_to_billing'] = df['isFraud'].apply(lambda x: distance_to_billing_gen(x))

In [22]:
print(df.head(20))

    step      type     amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0      1   PAYMENT    9839.64  C1231006815      170136.00       160296.36   
1      1   PAYMENT    1864.28  C1666544295       21249.00        19384.72   
2      1  TRANSFER     181.00  C1305486145         181.00            0.00   
3      1  CASH_OUT     181.00   C840083671         181.00            0.00   
4      1   PAYMENT   11668.14  C2048537720       41554.00        29885.86   
5      1   PAYMENT    7817.71    C90045638       53860.00        46042.29   
6      1   PAYMENT    7107.77   C154988899      183195.00       176087.23   
7      1   PAYMENT    7861.64  C1912850431      176087.23       168225.59   
8      1   PAYMENT    4024.36  C1265012928        2671.00            0.00   
9      1     DEBIT    5337.77   C712410124       41720.00        36382.23   
10     1     DEBIT    9644.94  C1900366749        4465.00            0.00   
11     1   PAYMENT    3099.97   C249177573       20771.00        17671.03   

In [25]:
# |orig_account_age|int|days|1-1200
# Uniform distribution for non-fraud, skewed towards 1 for fraud 
# Consistent to account (one account assigned one age)
# |dest_account_age|int|days|1-1200
# Same logic and consistent with orig_account_age

temp_df_1 = df[df['isFraud'] == 0]['nameOrig']
temp_df_2 = df[df['isFraud'] == 0]['nameDest']

no_fraud_names = pd.concat([temp_df_1, temp_df_2]).drop_duplicates()

temp_df_3 = df[df['isFraud'] == 1]['nameOrig']
temp_df_4 = df[df['isFraud'] == 1]['nameDest']

fraud_names = pd.concat([temp_df_3, temp_df_4]).drop_duplicates()

In [31]:
no_fraud_names = no_fraud_names.to_frame(name='name')
fraud_names = fraud_names.to_frame(name='name')

no_fraud_names['account_age'] = [np.random.default_rng().integers(low=1, high=1200, size=1)[0] for _ in no_fraud_names.index]
fraud_names['account_age'] = [int(np.random.default_rng().triangular(left=1, mode=1, right=1200, size=1)[0]) for _ in fraud_names.index]

print(no_fraud_names.head(20))
print(fraud_names.head(20))

           name  account_age
0   C1231006815          732
1   C1666544295           31
4   C2048537720          165
5     C90045638          337
6    C154988899          546
7   C1912850431          902
8   C1265012928          717
9    C712410124          431
10  C1900366749           23
11   C249177573          404
12  C1648232591          238
13  C1716932897            7
14  C1026483832          147
15   C905080434          194
16   C761750706          140
17  C1237762639          374
18  C2033524545          284
19  C1670993182          952
20    C20804602          821
21  C1566511282         1045
             name  account_age
2     C1305486145          144
3      C840083671          171
251   C1420196421          105
252   C2101527076          499
680    C137533655           29
681   C1118430673          197
724    C749981943          280
969   C1334405552          307
970    C467632528           23
1115  C1364127192          121
1116  C1635772897          435
1869   C669700766  

In [32]:
names = pd.merge(no_fraud_names, fraud_names, how='outer')
names = names.groupby('name').max()
print(names.head(20))

             account_age
name                    
C1000000639          705
C1000001337          360
C1000001725          555
C1000002591         1089
C1000003372            3
C1000003615          695
C1000004053          324
C1000004082          630
C1000004530          240
C1000004940          523
C1000005353          740
C1000005555          993
C1000005749           82
C1000005789           23
C1000006646          382
C1000006873          406
C1000008111         1034
C1000008236          250
C1000008393          415
C1000008582          588


In [33]:
# Saving out for safety, no need to run this cell in later runs
# df.to_parquet(os.path.join(PROJECT_ROOT, 'data', 'interim', 'checkpoint_1.parquet.gzip'))

In [34]:
print(len(names))
# Looks good

9073900


In [39]:
names = names.rename(columns={'name': 'nameOrig', 'account_age': 'orig_account_age'})
print(names.head())
df = pd.merge(df, names, how='left', left_on='nameOrig', right_on='name')
print(df.head(20))

             orig_account_age
name                         
C1000000639               705
C1000001337               360
C1000001725               555
C1000002591              1089
C1000003372                 3
    step      type     amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0      1   PAYMENT    9839.64  C1231006815      170136.00       160296.36   
1      1   PAYMENT    1864.28  C1666544295       21249.00        19384.72   
2      1  TRANSFER     181.00  C1305486145         181.00            0.00   
3      1  CASH_OUT     181.00   C840083671         181.00            0.00   
4      1   PAYMENT   11668.14  C2048537720       41554.00        29885.86   
5      1   PAYMENT    7817.71    C90045638       53860.00        46042.29   
6      1   PAYMENT    7107.77   C154988899      183195.00       176087.23   
7      1   PAYMENT    7861.64  C1912850431      176087.23       168225.59   
8      1   PAYMENT    4024.36  C1265012928        2671.00            0.00   
9      1     DEBIT  

In [40]:
names = names.rename(columns={'orig_account_age': 'dest_account_age'})
print(names.head())
df = pd.merge(df, names, how='left', left_on='nameDest', right_on='name')
print(df.head(20))

             dest_account_age
name                         
C1000000639               705
C1000001337               360
C1000001725               555
C1000002591              1089
C1000003372                 3
    step      type     amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0      1   PAYMENT    9839.64  C1231006815      170136.00       160296.36   
1      1   PAYMENT    1864.28  C1666544295       21249.00        19384.72   
2      1  TRANSFER     181.00  C1305486145         181.00            0.00   
3      1  CASH_OUT     181.00   C840083671         181.00            0.00   
4      1   PAYMENT   11668.14  C2048537720       41554.00        29885.86   
5      1   PAYMENT    7817.71    C90045638       53860.00        46042.29   
6      1   PAYMENT    7107.77   C154988899      183195.00       176087.23   
7      1   PAYMENT    7861.64  C1912850431      176087.23       168225.59   
8      1   PAYMENT    4024.36  C1265012928        2671.00            0.00   
9      1     DEBIT  

In [42]:
column_nan_count = df.isnull().sum()
print("NaN count per column:")
print(column_nan_count)

# sanity check to ensure all values are filled

NaN count per column:
step                   0
type                   0
amount                 0
nameOrig               0
oldbalanceOrg          0
newbalanceOrig         0
nameDest               0
oldbalanceDest         0
newbalanceDest         0
isFraud                0
isFlaggedFraud         0
user_agent             0
failed_attempts        0
distance_to_billing    0
orig_account_age       0
dest_account_age       0
dtype: int64


In [43]:
# |address_mismatch|int|NaN|boolean 1/0 for yes/no
# Generate randomly per account at 5% for non fraud and 13% for fraud 
# Flag if either orig or dest hits 1

no_fraud_names['address_mismatch'] = [stats.bernoulli.rvs(p = 0.05, size=1)[0] for _ in no_fraud_names.index]
fraud_names['address_mismatch'] = [stats.bernoulli.rvs(p = 0.13, size=1)[0] for _ in fraud_names.index]

# print(no_fraud_names.head())

          name  account_age  address_mismatch
0  C1231006815          732                 0
1  C1666544295           31                 0
4  C2048537720          165                 0
5    C90045638          337                 0
6   C154988899          546                 0


In [44]:
names = pd.merge(no_fraud_names, fraud_names, how='outer')
names = names.groupby('name').max()
# print(names.head(20))

             account_age  address_mismatch
name                                      
C1000000639          705                 0
C1000001337          360                 0
C1000001725          555                 0
C1000002591         1089                 0
C1000003372            3                 0
C1000003615          695                 1
C1000004053          324                 0
C1000004082          630                 0
C1000004530          240                 0
C1000004940          523                 0
C1000005353          740                 0
C1000005555          993                 0
C1000005749           82                 0
C1000005789           23                 0
C1000006646          382                 0
C1000006873          406                 0
C1000008111         1034                 0
C1000008236          250                 0
C1000008393          415                 0
C1000008582          588                 0


In [45]:
names = names.drop(columns=['account_age'])
names = names.rename(columns={'address_mismatch': 'orig_add'})
print(names.head())
df = pd.merge(df, names, how='left', left_on='nameOrig', right_on='name')

             account_age  orig_add
name                              
C1000000639          705         0
C1000001337          360         0
C1000001725          555         0
C1000002591         1089         0
C1000003372            3         0


In [59]:
names = names.rename(columns={'orig_add': 'dest_add'})
print(names.head())
df = pd.merge(df, names, how='left', left_on='nameDest', right_on='name')

             dest_add
name                 
C1000000639         0
C1000001337         0
C1000001725         0
C1000002591         0
C1000003372         0


In [61]:
def or_num(a, b):
    if a+b>=1:
        return 1
    else: 
        return 0

df['address_mismatch'] = df.apply(lambda x: or_num(x['dest_add'], x['orig_add']), axis=1)
print(df.head(20))

    step      type     amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0      1   PAYMENT    9839.64  C1231006815      170136.00       160296.36   
1      1   PAYMENT    1864.28  C1666544295       21249.00        19384.72   
2      1  TRANSFER     181.00  C1305486145         181.00            0.00   
3      1  CASH_OUT     181.00   C840083671         181.00            0.00   
4      1   PAYMENT   11668.14  C2048537720       41554.00        29885.86   
5      1   PAYMENT    7817.71    C90045638       53860.00        46042.29   
6      1   PAYMENT    7107.77   C154988899      183195.00       176087.23   
7      1   PAYMENT    7861.64  C1912850431      176087.23       168225.59   
8      1   PAYMENT    4024.36  C1265012928        2671.00            0.00   
9      1     DEBIT    5337.77   C712410124       41720.00        36382.23   
10     1     DEBIT    9644.94  C1900366749        4465.00            0.00   
11     1   PAYMENT    3099.97   C249177573       20771.00        17671.03   

In [62]:
df = df.drop(columns=['orig_add', 'dest_add'])
df.to_parquet(os.path.join(PROJECT_ROOT, 'data', 'processed', 'full_data.parquet.gzip'))

In [65]:
# boot test

test_df = pd.read_parquet(os.path.join(PROJECT_ROOT, 'data', 'processed', 'full_data.parquet.gzip'))
test_df.info()
print(test_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 19 columns):
 #   Column               Dtype  
---  ------               -----  
 0   step                 int64  
 1   type                 object 
 2   amount               float64
 3   nameOrig             object 
 4   oldbalanceOrg        float64
 5   newbalanceOrig       float64
 6   nameDest             object 
 7   oldbalanceDest       float64
 8   newbalanceDest       float64
 9   isFraud              int64  
 10  isFlaggedFraud       int64  
 11  user_agent           object 
 12  failed_attempts      int64  
 13  distance_to_billing  int64  
 14  orig_account_age     int64  
 15  dest_account_age     int64  
 16  orig_add             int64  
 17  dest_add             int64  
 18  address_mismatch     int64  
dtypes: float64(5), int64(10), object(4)
memory usage: 922.3+ MB
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C123